In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import accuracy_score, classification_report



In [2]:
df = pd.read_csv("C:/Users/hp/Desktop/Machine_Learning/data.csv")
print("Success! Data loaded.")

C:\Users\hp\AppData\Local\Temp\ipykernel_17940\1747373239.py:1: DtypeWarning: Columns (0: coupon_code, 1: additional) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("C:/Users/hp/Desktop/Machine_Learning/data.csv")


Success! Data loaded.


### 1. Dataset Preparation (Assuming df_clean is already prepared with features and target)


In [ ]:
df_clean = df.copy()

target_column = 'target' if 'target' in df_clean.columns else 'status'

df_clean = df_clean.dropna(subset=[target_column])

available_features = df_clean.columns.difference([target_column]).tolist()

X_raw = df_clean[available_features].copy()
y = df_clean[target_column]

num_cols = X_raw.select_dtypes(include=['number']).columns
X_raw[num_cols] = X_raw[num_cols].fillna(X_raw[num_cols].median())

cat_cols = X_raw.select_dtypes(include=['object', 'string', 'category']).columns
X_raw[cat_cols] = X_raw[cat_cols].fillna('Missing')
X = pd.get_dummies(X_raw, columns=cat_cols, drop_first=True)



### Train-Test Split (Full Data)


In [ ]:
X_train_full, X_test_full, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



### Stratified Sub-sampling (30k rows) for SVM & KNN compatibility


In [ ]:
sample_size = 30000
if len(X_train_full) > sample_size:
    _, X_train_sub, _, y_train_sub = train_test_split(
        X_train_full, y_train_full, test_size=sample_size/len(X_train_full), 
        random_state=42, stratify=y_train_full
    )
else:
    X_train_sub, y_train_sub = X_train_full, y_train_full



### Feature Scaling (Must for SVM, KNN, Logistic Regression)


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sub)
X_test_scaled = scaler.transform(X_test_full)



### ==========================================
### INITIALIZING ALL 5 MODELS (The Experts)
### ==========================================

In [ ]:

lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
knn_model = KNeighborsClassifier(n_neighbors=5, weights='distance', n_jobs=-1)
svm_model = SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=42)
dt_model = DecisionTreeClassifier(max_depth=12, min_samples_split=100, class_weight='balanced', random_state=42)
rf_model = RandomForestClassifier(n_estimators=100, max_depth=15, class_weight='balanced', n_jobs=-1, random_state=42)



### ==========================================
### CREATING THE CUSTOM ENSEMBLE (Voting Classifier)
### ==========================================

In [ ]:
custom_ensemble = VotingClassifier(
    estimators=[
        ('Logistic', lr_model),
        ('KNN', knn_model),
        ('SVM', svm_model),
        ('DecisionTree', dt_model),
        ('RandomForest', rf_model)
    ],
    voting='soft', 
    n_jobs=-1 # Uses all CPU cores to train models in parallel
)

print("Training Custom Ensemble (This might take a few minutes because SVM is included)...")
custom_ensemble.fit(X_train_scaled, y_train_sub)

y_pred_ensemble = custom_ensemble.predict(X_test_scaled)

print("=" * 50)
print(" CUSTOM ENSEMBLE (5-MODEL VOTING) PERFORMANCE ")
print("=" * 50)
print(f"Accuracy Score: {accuracy_score(y_test, y_pred_ensemble):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred_ensemble))
print("=" * 50)